In [ ]:
CREATE WAREHOUSE if not exists ECOMMERCE_WH
WAREHOUSE_SIZE = 'XSMALL'
AUTO_SUSPEND = 60
AUTO_RESUME = TRUE
INITIALLY_SUSPENDED = TRUE;

In [ ]:
use WAREHOUSE ECOMMERCE_WH;

In [ ]:
create database if not exists ecommerce_db;

In [ ]:
use database ecommerce_db;

In [ ]:
CREATE SCHEMA if not exists MART;


In [ ]:
create schema if not exists raw;

In [ ]:
create stage if not exists ECOM_STAGE;

In [ ]:
CREATE FILE FORMAT if not exists PARQUET_FORMAT
TYPE = PARQUET;

In [ ]:
USE DATABASE ECOMMERCE_DB;

In [ ]:
USE SCHEMA RAW;

In [ ]:
LIST @RAW.ECOM_STAGE;

In [ ]:
CREATE OR REPLACE FILE FORMAT my_parquet_format
TYPE = PARQUET;

In [ ]:
CREATE OR REPLACE TABLE RAW.USERS
USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
        INFER_SCHEMA(
            LOCATION => '@RAW.ECOM_STAGE/users_real.parquet',
            FILE_FORMAT => 'my_parquet_format'
        )
    )
);

In [ ]:
CREATE OR REPLACE TABLE RAW.PRODUCTS
USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
        INFER_SCHEMA(
            LOCATION => '@RAW.ECOM_STAGE/products_real.parquet',
            FILE_FORMAT => 'my_parquet_format'
        )
    )
);

In [ ]:
CREATE OR REPLACE TABLE RAW.ORDERS
USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
        INFER_SCHEMA(
            LOCATION => '@RAW.ECOM_STAGE/orders_real.parquet',
            FILE_FORMAT => 'my_parquet_format'
        )
    )
);

In [ ]:
CREATE OR REPLACE TABLE RAW.ORDER_ITEMS
USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
        INFER_SCHEMA(
            LOCATION => '@RAW.ECOM_STAGE/order_items_real.parquet',
            FILE_FORMAT => 'my_parquet_format'
        )
    )
);

In [ ]:
CREATE OR REPLACE TABLE RAW.REVIEWS
USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
        INFER_SCHEMA(
            LOCATION => '@RAW.ECOM_STAGE/reviews_real.parquet',
            FILE_FORMAT => 'my_parquet_format'
        )
    )
);

In [ ]:
CREATE OR REPLACE TABLE RAW.EVENTS
USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
        INFER_SCHEMA(
            LOCATION => '@RAW.ECOM_STAGE/events_real.parquet',
            FILE_FORMAT => 'my_parquet_format'
        )
    )
);

In [ ]:
COPY INTO RAW.USERS
FROM @RAW.ECOM_STAGE/users_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
COPY INTO RAW.EVENTS
FROM @RAW.ECOM_STAGE/events_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
COPY INTO RAW.REVIEWS
FROM @RAW.ECOM_STAGE/reviews_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
COPY INTO RAW.ORDER_ITEMS
FROM @RAW.ECOM_STAGE/order_items_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
COPY INTO RAW.ORDERS
FROM @RAW.ECOM_STAGE/orders_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
COPY INTO RAW.PRODUCTS
FROM @RAW.ECOM_STAGE/products_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
SELECT COUNT(*) FROM RAW.USERS;


In [ ]:
SELECT COUNT(*) FROM RAW.PRODUCTS;


In [ ]:
SELECT COUNT(*) FROM RAW.ORDERS;


In [ ]:
SELECT COUNT(*) FROM RAW.ORDER_ITEMS;


In [ ]:
SELECT COUNT(*) FROM RAW.REVIEWS;


In [ ]:
SELECT COUNT(*) FROM RAW.EVENTS;

In [ ]:
CREATE PIPE if not exists USERS_PIPE
AS
COPY INTO RAW.USERS
FROM @RAW.ECOM_STAGE/users_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
CREATE PIPE if not exists EVENTS_PIPE
AS
COPY INTO RAW.EVENTS
FROM @RAW.ECOM_STAGE/events_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
CREATE PIPE if not exists REVIEWS_PIPE
AS
COPY INTO RAW.REVIEWS
FROM @RAW.ECOM_STAGE/reviews_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


In [ ]:
CREATE PIPE if not exists ORDER_ITEMS_PIPE
AS
COPY INTO RAW.ORDER_ITEMS
FROM @RAW.ECOM_STAGE/order_items_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
CREATE PIPE if not exists ORDERS_PIPE
AS
COPY INTO RAW.ORDERS
FROM @RAW.ECOM_STAGE/orders_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
CREATE PIPE if not exists PRODUCTS_PIPE
AS
COPY INTO RAW.PRODUCTS
FROM @RAW.ECOM_STAGE/products_real.parquet
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
ALTER PIPE USERS_PIPE REFRESH;

In [ ]:
SELECT * 
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => 'USERS',
    START_TIME => DATEADD(HOUR, -1, CURRENT_TIMESTAMP())
));

In [ ]:
CREATE TABLE if not exists MART.FACT_SALES AS
SELECT 
    o."order_id",
    o."order_date",
    o."user_id",
    oi."product_id",
    oi."quantity",
    oi."item_price",
    oi."quantity" * oi."item_price" AS total_sales
FROM RAW.ORDERS o
JOIN RAW.ORDER_ITEMS oi
ON o."order_id" = oi."order_id";

In [ ]:
CREATE TABLE if not exists MART.DIM_USERS AS
SELECT * FROM RAW.USERS;

In [ ]:
CREATE TABLE if not exists MART.DIM_PRODUCTS AS
SELECT * FROM RAW.PRODUCTS;

In [ ]:
CREATE VIEW if not exists MART.REVENUE_BY_CATEGORY AS
SELECT
p."category",
SUM(f.total_sales) revenue
FROM MART.FACT_SALES f
JOIN MART.DIM_PRODUCTS p
ON f."product_id"=p."product_id"
GROUP BY p."category";

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("Ecommerce Sales Dashboard")

revenue = session.sql("""
SELECT SUM("quantity" * "item_price") AS total_revenue
FROM MART.FACT_SALES
""").to_pandas()
st.metric("Total Revenue", revenue.iloc[0,0])

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("Ecommerce Sales Dashboard")
category = session.sql("""
SELECT p."category", SUM(f."quantity" * f."item_price") revenue
FROM MART.FACT_SALES f
JOIN MART.DIM_PRODUCTS p
ON f."product_id" = p."product_id"
GROUP BY p."category"
""").to_pandas()

st.subheader("Sales by Category")
st.bar_chart(category.set_index("category"))

In [ ]:
import snowflake.connector
import pandas as pd

def get_data():
    conn = snowflake.connector.connect(
        user='YOUR_USER',
        password='YOUR_PASSWORD',
        account='YOUR_ACCOUNT',
        warehouse='YOUR_WAREHOUSE',
        database='YOUR_DATABASE',
        schema='YOUR_SCHEMA'
    )

    query = "SELECT * FROM SALES_DATA"  # your final transformed table
    df = pd.read_sql(query, conn)
    conn.close()
    return df


In [ ]:
import streamlit as st


import plotly.express as px
import snowflake.connector
import pandas as pd

# ✅ Cache for performance

def get_data():
    conn = snowflake.connector.connect(
        user='VEDU1611',
        password='YOUR_PASSWORD',
        account='fj50145.me-central2.gcp',
        warehouse='ECOMMERCE_WH',
        database='ECOMMERCE_DB',
        schema='MART'
    )

    try:
        query = "SELECT * FROM SALES_DATA"
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

# ✅ Load data
df = get_data()

st.title("📊 E-commerce Sales Dashboard")

# ✅ Check columns (prevents crash)
required_cols = ["REGION", "ORDER_STATUS", "SALES_AMOUNT", "ORDER_ID", "PRODUCT_NAME", "ORDER_DATE"]

for col in required_cols:
    if col not in df.columns:
        st.error(f"Missing column: {col}")
        st.stop()

# Sidebar filters
st.sidebar.header("Filters")

region = st.sidebar.multiselect(
    "Select Region",
    options=df["REGION"].dropna().unique(),
    default=df["REGION"].dropna().unique()
)

status = st.sidebar.multiselect(
    "Order Status",
    options=df["ORDER_STATUS"].dropna().unique(),
    default=df["ORDER_STATUS"].dropna().unique()
)

filtered_df = df[
    (df["REGION"].isin(region)) &
    (df["ORDER_STATUS"].isin(status))
]

# KPIs
total_sales = filtered_df["SALES_AMOUNT"].sum()
total_orders = filtered_df["ORDER_ID"].nunique()

col1, col2 = st.columns(2)
col1.metric("💰 Total Sales", f"{total_sales:,.0f}")
col2.metric("📦 Total Orders", total_orders)

# --- Top Selling Products ---
st.subheader("🏆 Top Selling Products")

top_products = (
    filtered_df.groupby("PRODUCT_NAME")["SALES_AMOUNT"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig1 = px.bar(
    top_products,
    x="PRODUCT_NAME",
    y="SALES_AMOUNT",
    title="Top 10 Products"
)

st.plotly_chart(fig1, use_container_width=True)

# --- Sales by Region ---
st.subheader("🌍 Sales by Region")

region_sales = (
    filtered_df.groupby("REGION")["SALES_AMOUNT"]
    .sum()
    .reset_index()
)

fig2 = px.pie(
    region_sales,
    names="REGION",
    values="SALES_AMOUNT",
    title="Sales Distribution"
)

st.plotly_chart(fig2, use_container_width=True)

# --- Order Status Trends ---
st.subheader("📈 Order Status Trends")

filtered_df["ORDER_DATE"] = pd.to_datetime(filtered_df["ORDER_DATE"])

status_trend = (
    filtered_df.groupby(["ORDER_DATE", "ORDER_STATUS"])
    .size()
    .reset_index(name="COUNT")
)

fig3 = px.line(
    status_trend,
    x="ORDER_DATE",
    y="COUNT",
    color="ORDER_STATUS",
    title="Order Status Over Time"
)

st.plotly_chart(fig3, use_container_width=True)

In [ ]:
SHOW SCHEMAS IN DATABASE ECOMMERCE_DB;